# Neural Response Extraction

Builds a per-image, per-area firing-rate matrix from the recorded session.

## Imports

In [1]:
import numpy as np
import pandas as pd
import pickle

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
from allensdk.brain_observatory.ecephys.ecephys_project_api import EcephysProjectWarehouseApi
from allensdk.brain_observatory.ecephys.ecephys_project_api.rma_engine import RmaEngine

C:\Users\neo\miniconda3\envs\allen216\lib\site-packages\pynwb\__init__.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
C:\Users\neo\miniconda3\envs\allen216\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the fixed image order from the previous notebook

In [3]:
labels_df = pd.read_csv("image_labels.csv")
image_names = labels_df["image_name"].tolist()
image_name_to_frame = dict(zip(labels_df["image_name"], labels_df["frame"].astype(float)))

print(f"Using {len(image_names)} images")

Using 118 images


## Load the session

In [4]:
data_directory = "./ecephys_cache_dir"
manifest_path = "./ecephys_cache_dir/manifest.json"

cache = EcephysProjectCache(
    manifest=manifest_path,
    fetch_api=EcephysProjectWarehouseApi(
        RmaEngine(
            scheme="http",
            host="api.brain-map.org",
            timeout=5 * 3600,
        )
    ),
)

SESSION_ID = 757216464
session = cache.get_session_data(SESSION_ID)

AREAS = ["LGd", "VISp", "VISrl", "VISam"]


C:\Users\neo\miniconda3\envs\allen216\lib\site-packages\hdmf\spec\namespace.py:535: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
C:\Users\neo\miniconda3\envs\allen216\lib\site-packages\hdmf\spec\namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.5.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


In [5]:
units = session.units

units_by_area = {}
for area in AREAS:
    area_units = units[units["ecephys_structure_acronym"] == area]
    units_by_area[area] = area_units.index.values
    print(f"{area}: {len(units_by_area[area])} units")

C:\Users\neo\miniconda3\envs\allen216\lib\site-packages\hdmf\spec\namespace.py:535: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
C:\Users\neo\miniconda3\envs\allen216\lib\site-packages\hdmf\spec\namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.5.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
C:\Users\neo\miniconda3\envs\allen216\lib\site-packages\hdmf\spec\namespace.py:535: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
C:\Users\neo\miniconda3\envs\allen216\lib\site-packages\hdmf\spec\namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.5.

LGd: 6 units
VISp: 85 units
VISrl: 37 units
VISam: 64 units


## Filter presentations to natural_scenes

In [10]:
presentations = session.stimulus_presentations
mask = (presentations["stimulus_name"] == "natural_scenes") & (presentations["frame"] != -1)
natural_scene_presentations = presentations.loc[mask]
print(len(natural_scene_presentations))

missing = set(image_name_to_frame.values()) - set(natural_scene_presentations["frame"].unique())
if missing:
    print(f"{len(missing)} frames not found in this session")
else:
    print("All frames found in this session's presentations.")

5900
All frames found in this session's presentations.


## Define the response time window

In [12]:
win_start = 0.05
win_end = 0.20
bin_edges = np.array([win_start, win_end])
window_duration = win_end - win_start

## Build the firing-rate matrix per area (118 * N)

In [13]:
neural_data = {"image_names": image_names, "areas": {}}

for area in AREAS:
    unit_ids = units_by_area[area]
    if len(unit_ids) == 0:
        print(f"Skipping {area}")
        continue

    n_images = len(image_names)
    n_units = len(unit_ids)
    response_matrix = np.zeros((n_images, n_units))

    for i, img_name in enumerate(image_names):
        frame = image_name_to_frame[img_name]
        img_presentations = natural_scene_presentations[natural_scene_presentations["frame"] == frame]
        presentation_ids = img_presentations.index.values

        if len(presentation_ids) == 0:
            print(f"{img_name} has 0 presentations")
            continue

        spike_counts = session.presentationwise_spike_counts(
            bin_edges=bin_edges,
            stimulus_presentation_ids=presentation_ids,
            unit_ids=unit_ids,
        )

        print(spike_counts)
        counts = spike_counts.sum(dim="time_relative_to_stimulus_onset").values
        firing_rates_hz = counts / window_duration
        mean_rate_per_unit = firing_rates_hz.mean(axis=0)

        response_matrix[i, :] = mean_rate_per_unit

    neural_data["areas"][area] = response_matrix
    print(f"{area}: response matrix shape {response_matrix.shape}")


<xarray.DataArray 'spike_counts' (stimulus_presentation_id: 50,
                                  time_relative_to_stimulus_onset: 1, unit_id: 6)>
array([[[ 2,  0,  3,  4,  1,  2]],

       [[ 0,  1,  4,  0,  7,  3]],

       [[ 0,  3,  6,  4,  2,  1]],

       [[ 2,  2,  6,  2,  1,  7]],

       [[ 2,  2,  6,  1,  6,  1]],

       [[ 1,  2,  0,  2,  0,  1]],

       [[ 1,  1,  8,  4,  2,  2]],

       [[ 1,  0,  1,  7,  2,  2]],

       [[ 0,  4,  7,  5,  4,  3]],

       [[ 0,  0,  1,  2,  1,  7]],

...

       [[ 0,  0,  0,  1,  2,  1]],

       [[ 1,  1,  4,  2, 10,  0]],

       [[ 0,  1,  0,  0,  0,  0]],

       [[ 0,  1,  4,  2,  3,  0]],

       [[ 2,  0,  2,  0,  6,  2]],

       [[ 2,  1,  4,  2,  6,  1]],

       [[ 1,  1,  1,  0,  3,  0]],

       [[ 0,  2,  0,  2,  3,  0]],

       [[ 2,  0,  4,  2, 11,  1]],

       [[ 0,  0,  1,  1, 10,  0]]], dtype=uint16)
Coordinates:
  * stimulus_presentation_id         (stimulus_presentation_id) int64 51503 ....
  * time_relative_to

## Save to disk

In [14]:
with open("neural_data.pkl", "wb") as f:
    pickle.dump(neural_data, f)